# Group G

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

In [ ]:
fig = px.histogram(df, y=df.Type, title="Conteggio totale per tipo di evento")
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
fig = px.scatter(df, y="LocationLat", x="LocationLng")
fig.update_layout(hovermode=False)
fig.show()

In [ ]:
#Selezionare eventi atmosferici storici che sono avvenuti nell'arco del tempo del nostro dataset e mostrarli con dei grafici (magari lo spostamento di un huragano (grafico animato))

#Grafico HeatMap che mostra l'andamento degli eventi nei vari mesi

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], utc=True)
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'], utc=True)

def to_local(group):
    tz_name = group.name 
    group['Local_Time'] = group['StartTime(UTC)'].dt.tz_convert(tz_name).dt.tz_localize(None)
    return group

df = df.groupby('TimeZone', group_keys=False).apply(to_local)

df['Local_Hour'] = df['Local_Time'].dt.hour

In [ ]:
hourly_counts = df.groupby(['Local_Hour', 'Type']).size().reset_index(name='Event_Count')

# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Event_Count', 
    color = 'Type',
    title = 'Distribuzione Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Event_Count': 'Numero di Eventi'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

In [ ]:
df["Year"] = df["StartTime(UTC)"].dt.year
df["Precipitation(mm)"] = df["Precipitation(in)"] * 25.4

mask = df["Type"] == "Rain"
annual_per_station = df.groupby(["Year", "AirportCode"])["Precipitation(mm)"].sum().reset_index()

final_yearly_data = annual_per_station.groupby("Year")["Precipitation(mm)"].mean().reset_index()


fig = px.bar(
    final_yearly_data,
    x = "Year",
    y = "Precipitation(mm)",
    title="Precipitazioni Medie Annuali per Stazione negli USA",
    labels={
        "Precipitation(mm)": "Media Pioggia Annua [mm]", 
        "Year": "Anno"
    },
    template = "plotly_white"
    )

# non mi convince questo grafico
fig.show()